In [24]:

from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
            model="qwen/qwen3-4b",
            temperature=0.2,
            top_p=0.8,
            api_key="lm-studio",  # Replace with your actual API key
            base_url="http://localhost:1234/v1"  # Replace with your actual base URL
        )

In [25]:
async def generate_answer(llm, question, contexts):
    combined_context = "\n\n".join(contexts)

    prompt = f"""
        Bạn là một trợ lý y tế thông minh, chỉ trả lời các câu hỏi liên quan đến y tế. Dưới đây là các câu hỏi từ người dùng và ngữ cảnh được cung cấp từ cơ sở dữ liệu:

        Câu hỏi:
        {question}

        Ngữ cảnh:
        {combined_context}

        Dựa trên câu hỏi và ngữ cảnh, hãy tổng hợp và đưa ra một câu trả lời rõ ràng, chính xác. Nếu không có đủ thông tin trong ngữ cảnh, hãy trả lời dựa trên kiến thức chung của bạn. 
        Trả lời bằng tiếng Việt và format câu trả lời theo dạng markdown một cách dễ đọc, không có các ký tự khoảng trắng thừa.
        """

    response = await llm.ainvoke(prompt)

    return response.content.strip()

In [26]:
import json
import asyncio
from tqdm import tqdm

INPUT_FILE = "rag_eval_dataset_100.jsonl"
OUTPUT_FILE = "rag_eval_dataset_with_llm.jsonl"

In [27]:
async def process_file(llm):

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:

        for item in tqdm(records):

            question = item["question"]
            contexts = item["contexts"]

            try:
                llm_answer = await generate_answer(
                    llm,
                    question,
                    contexts
                )

            except Exception as e:
                print(f"Error: {question}")
                print(e)

                llm_answer = ""

            item["llm_answer"] = llm_answer

            out_f.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                )
                + "\n"
            )

    print("Done")

In [28]:
await process_file(llm)

100%|██████████| 100/100 [26:49<00:00, 16.09s/it]

Done


Evaluate Blue, Rouge, Similarity

In [29]:
import json
import pandas as pd
import numpy as np

from tqdm import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
# =========================
# CONFIG
# =========================

INPUT_FILE = "rag_eval_dataset_with_llm.jsonl"

DETAIL_OUTPUT = "evaluation_detail.csv"
SUMMARY_OUTPUT = "evaluation_summary.csv"

EMBED_MODEL = "bkai-foundation-models/vietnamese-bi-encoder"

In [31]:
# =========================
# LOAD MODEL
# =========================

print("Loading embedding model...")

embed_model = SentenceTransformer(
    EMBED_MODEL,
    trust_remote_code=True,
    device="cuda"
)

rouge = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False
)

smooth = SmoothingFunction().method1


Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4096.32it/s]


In [32]:
# =========================
# FUNCTIONS
# =========================

def calc_rouge(reference, candidate):
    scores = rouge.score(reference, candidate)

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def calc_bleu(reference, candidate):

    try:
        return sentence_bleu(
            [reference.split()],
            candidate.split(),
            smoothing_function=smooth
        )
    except:
        return 0.0


def calc_similarity(text1, text2):

    emb = embed_model.encode(
        [text1, text2],
        normalize_embeddings=True
    )

    sim = cosine_similarity(
        [emb[0]],
        [emb[1]]
    )[0][0]

    return float(sim)

In [33]:
# =========================
# READ FILE
# =========================

records = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Total records:", len(records))

# =========================
# EVALUATE
# =========================

results = []

for item in tqdm(records):

    question = item.get("question", "")

    answer = item.get("answer", "")

    llm_answer = item.get("llm_answer", "")

    contexts = item.get("contexts", [])

    context_text = "\n".join(contexts)

    # ===================================
    # CONTEXT vs ANSWER
    # ===================================

    context_rouge = calc_rouge(
        answer,
        context_text
    )

    context_bleu = calc_bleu(
        answer,
        context_text
    )

    context_sim = calc_similarity(
        answer,
        context_text
    )

    # ===================================
    # LLM ANSWER vs ANSWER
    # ===================================

    answer_rouge = calc_rouge(
        answer,
        llm_answer
    )

    answer_bleu = calc_bleu(
        answer,
        llm_answer
    )

    answer_sim = calc_similarity(
        answer,
        llm_answer
    )

    results.append({

        "question": question,

        # ---------- Context ----------
        "context_rouge1": context_rouge["rouge1"],
        "context_rouge2": context_rouge["rouge2"],
        "context_rougeL": context_rouge["rougeL"],
        "context_bleu": context_bleu,
        "context_similarity": context_sim,

        # ---------- LLM ----------
        "answer_rouge1": answer_rouge["rouge1"],
        "answer_rouge2": answer_rouge["rouge2"],
        "answer_rougeL": answer_rouge["rougeL"],
        "answer_bleu": answer_bleu,
        "answer_similarity": answer_sim,
    })


Total records: 100


100%|██████████| 100/100 [00:19<00:00,  5.14it/s]


In [34]:
# =========================
# SAVE DETAIL
# =========================

detail_df = pd.DataFrame(results)

detail_df.to_csv(
    DETAIL_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", DETAIL_OUTPUT)

# =========================
# SUMMARY
# =========================

summary = {}

for col in detail_df.columns:

    if col == "question":
        continue

    summary[col] = detail_df[col].mean()

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", SUMMARY_OUTPUT)

print("\n========== SUMMARY ==========")

for k, v in summary.items():
    print(f"{k:25s}: {v:.4f}")

Saved: evaluation_detail.csv
Saved: evaluation_summary.csv

========== SUMMARY ==========
context_rouge1           : 0.4143
context_rouge2           : 0.3583
context_rougeL           : 0.3385
context_bleu             : 0.1882
context_similarity       : 0.7766
answer_rouge1            : 0.6387
answer_rouge2            : 0.3859
answer_rougeL            : 0.3511
answer_bleu              : 0.0833
answer_similarity        : 0.6596


Evaluate LLM jugde

In [35]:
def build_context_eval_prompt(question, answer, contexts):

    context_text = "\n\n".join(contexts)

    return f"""
Bạn là chuyên gia đánh giá hệ thống RAG.

Câu hỏi:
{question}

Đáp án chuẩn:
{answer}

Context được truy xuất:
{context_text}

Hãy đánh giá:

1. Context Relevance (1-10):
Context có liên quan đến câu hỏi không?

2. Context Completeness (1-10):
Context có chứa đủ thông tin để sinh ra đáp án chuẩn không?

3. Context Precision (1-10):
Context có ít thông tin nhiễu không?

4. Context Recall (1-10):
Context có bỏ sót thông tin quan trọng của đáp án không?

Trả về JSON:

{{
    "context_relevance": number,
    "context_completeness": number,
    "context_precision": number,
    "context_recall": number,
    "reason": "giải thích ngắn"
}}
"""

In [36]:
def build_answer_eval_prompt(
        question,
        answer,
        llm_answer
):

    return f"""
Bạn là chuyên gia đánh giá chatbot y tế.

Câu hỏi:
{question}

Đáp án chuẩn:
{answer}

Câu trả lời chatbot:
{llm_answer}

Đánh giá:

1. Correctness (1-10)
2. Completeness (1-10)
3. Relevance (1-10)
4. Hallucination (1-10)

Hallucination:
10 = hoàn toàn không bịa
1 = bịa rất nhiều

Trả về JSON:

{{
    "correctness": number,
    "completeness": number,
    "relevance": number,
    "hallucination": number,
    "reason": "giải thích ngắn"
}}
"""

In [37]:
import json
from tqdm import tqdm

results = []

with open("rag_eval_dataset_with_llm.jsonl", "r", encoding="utf-8") as f:

    for line in tqdm(f):

        item = json.loads(line)

        question = item["question"]
        answer = item["answer"]
        contexts = item["contexts"]
        llm_answer = item["llm_answer"]

        # ====================
        # Context Judge
        # ====================

        context_prompt = build_context_eval_prompt(
            question,
            answer,
            contexts
        )

        context_eval = llm.invoke(
            context_prompt
        ).content

        # ====================
        # Answer Judge
        # ====================

        answer_prompt = build_answer_eval_prompt(
            question,
            answer,
            llm_answer
        )

        answer_eval = llm.invoke(
            answer_prompt
        ).content

        results.append({
            "question": question,
            "context_eval": context_eval,
            "answer_eval": answer_eval
        })

20it [07:22, 22.11s/it]


BadRequestError: Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 5694>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}

In [ ]:
import json
from collections import defaultdict

context_metrics = defaultdict(list)
answer_metrics = defaultdict(list)

for item in results:
    try:
        context_eval = json.loads(item["context_eval"])
        answer_eval = json.loads(item["answer_eval"])

        # Context metrics
        for key in [
            "context_relevance",
            "context_completeness",
            "context_precision",
            "context_recall"
        ]:
            if key in context_eval:
                context_metrics[key].append(context_eval[key])

        # Answer metrics
        for key in [
            "correctness",
            "completeness",
            "relevance",
            "hallucination"
        ]:
            if key in answer_eval:
                answer_metrics[key].append(answer_eval[key])

    except Exception as e:
        print("Parse error:", e)

In [ ]:
print("\n===== CONTEXT RETRIEVAL EVALUATION =====")

for metric, values in context_metrics.items():
    avg = sum(values) / len(values)
    print(f"{metric}: {avg:.2f}")

overall_context = (
    sum(sum(v) for v in context_metrics.values())
    / sum(len(v) for v in context_metrics.values())
)

print(f"\nOverall Context Score: {overall_context:.2f}/10")


print("\n===== ANSWER GENERATION EVALUATION =====")

for metric, values in answer_metrics.items():
    avg = sum(values) / len(values)
    print(f"{metric}: {avg:.2f}")

overall_answer = (
    sum(sum(v) for v in answer_metrics.values())
    / sum(len(v) for v in answer_metrics.values())
)

print(f"\nOverall Answer Score: {overall_answer:.2f}/10")


===== CONTEXT RETRIEVAL EVALUATION =====
context_relevance: 9.30
context_completeness: 8.80
context_precision: 8.10
context_recall: 9.50

Overall Context Score: 8.93/10

===== ANSWER GENERATION EVALUATION =====
correctness: 8.60
completeness: 7.80
relevance: 9.30
hallucination: 9.70

Overall Answer Score: 8.85/10
